In [ ]:
# ------------------------------------------------------------
# Step 2.0: Notebook Initialization
# ------------------------------------------------------------

import sys
sys.path.append("..")

from src.config import *

# Load & register data
df_raw = load_raw_data(url="URL")    # Paste the URL if the downloaded file is not present on disk
register_duckdb_table(df_raw, table_name="trips")

print("Config loaded successfully.")

print("\n" + "=" * 70)
print("Baseline Dataset")
print("=" * 70)

print(f"   Rows    : {df_raw.shape[0]:,}")
print(f"   Columns : {df_raw.shape[1]}")
print(f"\n   Columns : {df_raw.columns.tolist()}")

## Step 2.1: Target Variable Analysis
Analyse the distribution of `fare_amount`, the target variable for our regression model.
Understanding its shape, skewness, and spread is critical for choosing the right model and loss function.

In [ ]:
# ------------------------------------------------------------
# Step 2.1: Target Variable Analysis
# ------------------------------------------------------------

def analyze_target(df:pd.DataFrame, target_col: str = TARGET_COL) -> dict:
    """
    Compute descriptive statistics for the target variable.

    Parameters
    ----------
    df: pd.DataFrame
        The raw DataFrame.
    target_col: str
        Name of the target column (default ``TARGET_COL``).

    Returns
    -------
    dict
        Key statistics: mean, median, std, skewness, kurtosis,
        and percentile boundaries.
    """
    series = df[target_col].dropna()

    stats = {
        "count": len(series),
        "mean": round(series.mean(), 2),
        "median": round(series.median(), 2),
        "std": round(series.std(), 2),
        "min": round(series.min(), 2),
        "max": round(series.max(), 2),
        "skewness": round(series.skew(), 4),
        "kurtosis": round(series.kurtosis(), 4),
        "p25": round(series.quantile(0.25), 2),
        "p75": round(series.quantile(0.75), 2),
        "p95": round(series.quantile(0.95), 2),
        "p99": round(series.quantile(0.99), 2),
    }
    for k, v in stats.items():
        logger.info(f"  {k:<12}: {v:,}")

    return stats

def plot_target_distribution(
    df: pd.DataFrame,
    target_col: str = TARGET_COL
) -> None:
    """
    Plot the distribution of the target variable using three views:
        1. Full distribution histogram with KDE
        2. Clipped distribution (p1-p99) to remove extreme outliers
        3. Log-transformed distribution to assess normality after transform

    Parameters
    ----------
    df: pd.DataFrame
        The raw DataFrame.
    target_col: str
        Name of the target column.
    """
    series = df[target_col].dropna()
    p1 = series.quantile(0.01)
    p99 = series.quantile(0.99)
    clipped = series.clip(lower=p1, upper=p99)
    log_series = np.log1p(series.clip(lower=0))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Target Variable: fare_amount Distribution", fontsize=15)

    # Plot 1: Full distribution
    axes[0].hist(
        series, bins=100,
        color=PALETTE["primary"], edgecolor="white", linewidth=0.3
    )
    axes[0].axvline(series.mean(), color=PALETTE["accent"], linestyle="--", label=f"Mean ${series.mean():.2f}")
    axes[0].axvline(series.median(), color=PALETTE["secondary"], linestyle="--", label=f"Median ${series.median():.2f}")
    axes[0].set_title("Full Distribution")
    axes[0].set_xlabel("Fare Amount ($)")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()

    # Plot 2: Clipped (p1-p99)
    axes[1].hist(
        clipped, bins=100,
        color=PALETTE["secondary"], edgecolor="white", linewidth=0.3
    )
    axes[1].axvline(clipped.mean(), color=PALETTE["accent"], linestyle="--", label=f"Mean ${clipped.mean():.2f}")
    axes[1].axvline(clipped.median(), color=PALETTE["primary"], linestyle="--", label=f"Median ${clipped.median():.2f}")
    axes[1].set_title("Clipped Distribution (p1-p99)")
    axes[1].set_xlabel("Fare Amount ($)")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()

    # Plot 3: Log1p transformed
    axes[2].hist(
        log_series, bins=100,
        color=PALETTE["neutral"], edgecolor="white", linewidth=0.3
    )
    axes[2].axvline(log_series.mean(), color=PALETTE["accent"], linestyle="--", label=f"Mean ${log_series.mean():.2f}")
    axes[2].axvline(log_series.median(), color=PALETTE["secondary"], linestyle="--", label=f"Median ${log_series.median():.2f}")
    axes[2].set_title("Log1p Transformed Distribution")
    axes[2].set_xlabel("log1p(Fare Amount)")
    axes[2].set_ylabel("Frequency")
    axes[2].legend()

    plt.tight_layout()
    save_figure(fig, "phase2_target_distribution")
    plt.show()

# Run analysis
print("=" * 70, flush=True)
print("Target Variable Statistics", flush=True)
print("=" * 70, flush=True)
target_stats = analyze_target(df_raw)

print("\n" + "=" * 70)
print("Percentile Boundaries")
print("=" * 70)
print(f"   p25 : ${target_stats['p25']}")
print(f"   p75 : ${target_stats['p75']}")
print(f"   p95 : ${target_stats['p95']}")
print(f"   p99 : ${target_stats['p99']}")
print(f"   Max : ${target_stats['max']}")

print("\n" + "=" * 70)
print("Skewness & Kurtosis")
print("=" * 70)
print(f"   Skewness : {target_stats['skewness']}  (>1 = right skewed)")
print(f"   Kurtosis : {target_stats['kurtosis']}  (>3 = heavy tails)")

plot_target_distribution(df_raw)


## Step 2.1: Target Variable Analysis ✅

### Descriptive Statistics

| Metric | Value |
|---|---|
| Count | $1,000,000$ |
| Mean | $\$13.15$ |
| Median | $\$9.50$ |
| Std Dev | $\$11.87$ |
| Min | $-\$233.00$ |
| Max | $\$621.50$ |
| Skewness | $4.2194$ |
| Kurtosis | $60.0893$ |

### Percentile Boundaries

| Percentile | Value |
|---|---|
| p25 | $\$6.50$ |
| p75 | $\$14.50$ |
| p95 | $\$37.50$ |
| p99 | $\$52.00$ |
| Max | $\$621.50$ |

### Plot Observations

**Plot 1: Full Distribution:**
- The bulk of trips are compressed in the $\$0–\$50$ range, making the distribution appear almost flat due to extreme outliers stretching to $\$621.50$ and negative values down to $-\$233$
- Negative fares are clearly invalid and will be removed in Phase 3

**Plot 2: Clipped Distribution (p1–p99):**
- After removing the extreme 1\% tails, the distribution is strongly **right-skewed** with a peak around $\$6–\$8$
- The mean ($\$12.98$) is pulled significantly right of the median ($\$9.50$) confirming the skew is driven by high-fare outlier trips
- A notable spike appears at the $\$52$ boundary (the p99 clip point) representing a concentration of airport/long-distance fares
- The long right tail suggests a small but consistent volume of high-value trips (airport runs, outer-borough destinations)

**Plot 3: Log1p Transformed Distribution:**
- The log1p transform substantially reduces skewness, producing a more bell-shaped distribution centred around $2.35–2.44$
- The distribution is still mildly right-skewed with a secondary cluster visible in the 4–7 range, corresponding to the high-fare
  outlier trips
- The transform is not perfect but is a significant improvement over the raw distribution

### Key Modelling Implications
- **Right skew (4.22) and extreme kurtosis (60.09)** mean raw `fare_amount` will violate linear regression assumptions
- **Log1p transformation** of the target variable will be applied in Phase 4 to stabilise variance and improve model performance
- The **IQR of $\$8.00$** ($\$6.50–\$14.50$) tells us $50\%$ of all trips fall within a narrow, predictable fare band, a good signal for
  regression
- Negative fares (min = $-\$233$) are data quality issues already flagged in the Phase 1 data contract for removal in Phase 3

## Step 2.2: Numeric Feature Distributions
Visualise the distribution of all numeric features. This helps identify skewness, multimodal distributions, and columns that may benefit from transformation before modelling.

In [ ]:
# ------------------------------------------------------------
# Step 2.2: Numeric Feature Distribution
# ------------------------------------------------------------

def get_numeric_features(
    df: pd.DataFrame,
    exclude: list[str] = None
) -> list[str]:
    """
    Return a list of numeric column names, excluding specified columns.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to inspect.
    exclude: list[str], optional
        Column names to exclude (e.g. target, ID columns).

    Returns
    -------
    list[str]
        Sorted list of numeric column names.
    """
    exclude = exclude or []
    return [
        col for col in df.select_dtypes(include=[np.number]).columns
        if col not in exclude
    ]

def compute_distribution_stats(
    df: pd.DataFrame,
    cols: list[str],
) -> pd.DataFrame:
    """
    Compute skewness and kurtosis for a list of numeric columns.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to analyze.
    cols: list[str]
        Numeric columns to include.

    Returns
    -------
    pd.DataFrame
        Table with mean, std, skewness and kurtosis per column,
        sorted by absolute skewness descending.
    """
    records = []
    for col in cols:
        series = pd.to_numeric(df[col], errors="coerce").dropna()
        records.append({
            "column": col,
            "mean": round(series.mean(), 2),
            "std": round(series.std(), 2),
            "skewness": round(series.skew(), 4),
            "kurtosis": round(series.kurtosis(), 4),
        })

    return (
        pd.DataFrame(records)
        .sort_values("skewness", key=abs, ascending=False)
        .reset_index(drop=True)
    )

def plot_numeric_distributions(
    df: pd.DataFrame,
    cols: list[str],
    n_cols: int = 3
) -> None:
    """
    Plot histograms with KDE overlays for all numeric features.

    Each subplot shows the distribution of one feature with vertical
    lines marking the mean and median for quick visual comparison.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to plot.
    cols: list[str]
        Numeric columns to include.
    n_cols: int, optional
        Number of subplot columns in the grid (default 3).
    """
    n_rows = int(np.ceil(len(cols) / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 6, n_rows * 4)
    )
    axes = axes.flatten()

    for i, col in enumerate(cols):
        series = pd.to_numeric(df[col], errors="coerce").dropna()

        # Clip to p1-p99 for readability
        p1, p99 = series.quantile(0.01), series.quantile(0.99)
        clipped = series.clip(lower=p1, upper=p99)

        axes[i].hist(
            clipped, bins=60,
            color=PALETTE["primary"], edgecolor="white", linewidth=0.3,
            density=True
        )

        # KDE overlay
        #clipped.plot.kde(ax=axes[i], color=PALETTE["accent"], linewidth=1.5)
        # KDE overlay (skip if data is constant or nearly constant)
        if clipped.var() > 1e-32:   # small tolerance for numerical stability
            try:
                clipped.plot.kde(ax=axes[i], color=PALETTE["accent"], linewidth=1.5)
            except np.linalg.LinAlgError as l:
                # Fallback: skip KDE silently (or print a warning)
                logger.warning(f"KDE error: {l}")

        # Mean & median lines
        axes[i].axvline(
            clipped.mean(), color=PALETTE["accent"],
            linestyle="--", linewidth=1.2,
            label=f"Mean {clipped.mean():.2f}"
        )
        axes[i].axvline(
            clipped.median(), color=PALETTE["secondary"],
            linestyle="--", linewidth=1.2,
            label=f"Median {clipped.median():.2f}"
        )

        axes[i].set_title(col)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("Density")
        axes[i].legend(fontsize=8)

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Numeric Feature Distributions (clipped p1-p99)", fontsize=15)
    plt.tight_layout()
    save_figure(fig, "phase2_numeric_distributions")
    plt.show()

# Run
# Exclude target, ID-like and location columns
EXCLUDE_COLS = [
    TARGET_COL, "id", "pulocationid", "dolocationid"
]

numeric_cols = get_numeric_features(df_raw, exclude=EXCLUDE_COLS)
logger.info(f"Numeric features to analyze: {numeric_cols}")

print("\n" + "=" * 70)
print("Distribution Statistics")
print("=" * 70)

dist_stats = compute_distribution_stats(df_raw, numeric_cols)
print(dist_stats.to_string(index=False))

plot_numeric_distributions(df_raw, numeric_cols)


## Step 2.2: Numeric Feature Distributions ✅

### Distribution Statistics

| Column | Mean | Std | Skewness | Kurtosis | Assessment |
|---|---|---|---|---|---|
| `ratecodeid` | 1.05 | 0.50 | 117.36 | 22,660.67 | Extreme spike, ~99\% are code 1 |
| `improvement_surcharge` | 0.30 | 0.02 | -36.31 | 1,378.26 | Near-constant, fixed at \$0.30 |
| `mta_tax` | 0.50 | 0.04 | -14.27 | 246.48 | Near-constant, fixed at \$0.50 |
| `tolls_amount` | 0.36 | 1.61 | 11.02 | 729.35 | Zero-inflated, most trips have no tolls |
| `tip_amount` | 1.74 | 2.58 | 6.06 | 157.42 | Zero-inflated, cash tips not recorded |
| `trip_distance` | 3.12 | 3.95 | 3.21 | 21.05 | Right skewed, few very long trips |
| `passenger_count` | 1.66 | 1.28 | 2.14 | 3.61 | Right skewed, mostly solo riders |
| `payment_type` | 1.37 | 0.50 | 0.87 | -0.07 | Near-binary, credit card vs cash |
| `vendorid` | 1.56 | 0.50 | -0.25 | -1.94 | Balanced binary, two vendors |

### Plot Observations (clipped p1–p99)

**`vendorid`**: Perfect bimodal distribution with two sharp symmetric spikes at values 1 and 2, confirming exactly two vendors in the dataset. Mean (1.56) sits between the two spikes. No transformation needed, will be one-hot encoded.

**`passenger_count`**: Dominant spike at 1 passenger with progressively smaller spikes at 2, 3, 4, 5, and 6. The KDE curve clearly shows a multimodal but heavily solo-rider-dominated distribution. Mean (1.66) is pulled right by group rides. Will be binned into categories in Phase 4.

**`trip_distance`**: Sharp spike near 0–2 miles followed by a long right tail extending to ~25 miles after clipping. Mean (3.09) is nearly double the median (1.70), confirming strong right skew driven by occasional long trips. One of the strongest expected fare predictors, log1p transformation will be applied in Phase 4.

**`ratecodeid`**: Near-total spike at code 1 (Standard rate) with a barely visible secondary spike at code 2 (JFK). Skewness of 117.36 is by far the highest in the dataset. Treating this as a numeric feature would mislead any model, will be one-hot encoded into binary flags in Phase 4.

**`payment_type`**: Two dominant spikes at 1 (Credit Card) and 2 (Cash) with negligible representation of other types. Behaves as a near-binary variable. Will be encoded as a binary flag in Phase 4.

**`mta_tax`**: Single near-perfect spike at \$0.50 with almost no variance (std = 0.04). This is a fixed regulatory charge with virtually no predictive signal. Strong candidate for dropping in Phase 4.

**`tip_amount`**: Large spike at $\$0$ followed by a right-skewed distribution extending to ~\$12–15. The zero spike is artificially inflated because cash tips are not recorded by the meter. The KDE confirms a two-component
distribution (zero vs non-zero). log1p transformation recommended in Phase 4.

**`tolls_amount`**: Extreme zero-inflation with a dominant spike at $\$0$ and a tiny secondary spike around \$5–6 (likely bridge/tunnel tolls). The vast majority of NYC trips incur no tolls. log1p transformation recommended.

**`improvement_surcharge`**: Single spike at exactly $\$0.30$ with essentially zero variance (std = 0.02). Fixed regulatory surcharge with no predictive power. Strong candidate for dropping in Phase 4.

### Key Feature Engineering Implications for Phase 4

| Column | Action | Reason |
|---|---|---|
| `ratecodeid` | One-hot encode | Categorical codes, not numeric |
| `improvement_surcharge` | Drop | Near-zero variance, no signal |
| `mta_tax` | Drop | Near-zero variance, no signal |
| `tolls_amount` | log1p transform | Zero-inflated, right skewed |
| `tip_amount` | Use with caution | Zero-inflated by payment type |
| `trip_distance` | log1p transform | Strong predictor, right skewed |
| `passenger_count` | Bin into groups | Multimodal, not truly continuous |
| `payment_type` | Binary flag | Near-binary distribution |
| `vendorid` | One-hot encode | True binary categorical |

## Step 2.3: Categorical Feature Distributions
Analyse columns that represent discrete categories, both true object dtype columns and numeric columns that encode categorical meaning. Understanding their value distributions informs encoding strategies in Phase 4.

In [ ]:
# ------------------------------------------------------------
# Step 2.3: Categorical Feature Distribution
# ------------------------------------------------------------

# Columns that are categorical in meaning regardless of dtype
CATEGORICAL_COLS: list[str] = [
    "vendorid",
    "ratecodeid",
    "payment_type",
    "passenger_count",
    "store_and_fwd_flag",
]

# Human-readable labels for each column's values
CATEGORY_LABELS: dict[str, dict] = {
    "vendorid": {
        1: "Creative Mobile",
        2: "VeriFone",
    },
    "ratecodeid": {
        1: "Standard",
        2: "JFK",
        3: "Newark",
        4: "Nassau/Westchester",
        5: "Negotiated",
        6: "Group Ride",
        99: "Invalid",
    },
    "payment_type": {
        1: "Credit Card",
        2: "Cash",
        3: "No Charge",
        4: "Dispute",
        5: "Unknown",
        6: "Voided",
    },
    "passenger_count"   : None,  # use raw numeric values
    "store_and_fwd_flag": None,  # use raw string values
}

def compute_categorical_stats(
    df: pd.DataFrame,
    cols: list[str]
) -> dict[str, pd.DataFrame]:
    """
    Compute value counts and percentage share for each categorical column.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to analyze.
    cols: list[str]
        Categorical columns to summarize.

    Returns
    -------
    dict[str, pd.DataFrame]
        Mapping of column name -> frequency table with columns:
        value, label, count, pct.
    """
    summaries = {}
    for col in cols:
        vc = df[col].value_counts(dropna=False).reset_index()
        vc.columns = ["value", "count"]
        vc["pct"] = (vc["count"] / len(df) * 100).round(2)
        
        # Apply human-readable labels if defined
        label_map = CATEGORY_LABELS.get(col)
        vc["label"] = (
            vc["value"].map(label_map)
            if label_map else vc["value"].astype(str)
        )
        summaries[col] = vc[["value", "label", "count", "pct"]]

    return summaries

def plot_categorical_distributions(
    summaries: dict[str, pd.DataFrame],
    n_cols: int = 3
) -> None:
    """
    Plot bar charts for each categorical feature's value distribution.

    Bars are annotated with percentage labels. Columns with more than 8
    unique values use horizontal bars for readability.

    Parameters
    ----------
    summaries: dict[str, pd.DataFrame]
        Output of ``compute_categorical_stats()``.
    n_cols: int, optional
        Number of subplot columns in the grid (default 3).
    """
    n_features = len(summaries)
    n_rows = int(np.ceil(n_features / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 6, n_rows * 4)
    )
    axes = axes.flatten()

    colors = [
        PALETTE["primary"],
        PALETTE["secondary"],
        PALETTE["accent"],
        PALETTE["neutral"],
    ]

    for i, (col, freq_df) in enumerate(summaries.items()):
        bar_colors = [colors[j % len(colors)] for j in range(len(freq_df))]

        bars = axes[i].bar(
            freq_df["label"].astype(str),
            freq_df["count"],
            color=bar_colors,
            edgecolor="white",
            linewidth=0.4
        )

        # Annotate each bar with its percentage
        for bar, (_, row) in zip(bars, freq_df.iterrows()):
            axes[i].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + freq_df["count"].max() * 0.01,
                f"{row['pct']}%",
                ha="center", va="bottom", fontsize=8
            )

        axes[i].set_title(col)
        axes[i].set_ylabel("Count")
        axes[i].set_xlabel("")
        axes[i].yaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f"{x/1_000:.0f}K")
        )
        axes[i].tick_params(axis="x", rotation=25)

    # Hide any unused subplot panels
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Categorical Feature Distributions", fontsize=15)
    plt.tight_layout()
    save_figure(fig, "phase2_categorical_distributions")
    plt.show()

# Run
cat_summaries = compute_categorical_stats(df_raw, CATEGORICAL_COLS)

print("\n" + "=" * 70)
print("Categorical Feature Summaries")
print("=" * 70)
for col, freq_df in cat_summaries.items():
    print(f"\n{'-' * (45 - len(col))} {col} {'-' * (45 - len(col))}")
    print(freq_df.to_string(index=False))

plot_categorical_distributions(cat_summaries)


## Step 2.3: Categorical Feature Distributions ✅

### Value Frequency Tables

**`vendorid`: Vendor Split**

| Vendor | Count | % |
|---|---|---|
| VeriFone Inc. | 561,929 | 56.19% |
| Creative Mobile Tech | 438,071 | 43.81% |

**`ratecodeid`: Rate Code Distribution**

| Rate Code | Count | % |
|---|---|---|
| Standard | 967,313 | 96.73% |
| JFK | 25,555 | 2.56% |
| Negotiated | 3,981 | 0.40% |
| Newark | 2,456 | 0.25% |
| Nassau/Westchester | 670 | 0.07% |
| Invalid (99) | 15 | 0.00% |
| Group Ride | 10 | 0.00% |

**`payment_type`: Payment Method**

| Payment Type | Count | % |
|---|---|---|
| Credit Card | 638,576 | 63.86% |
| Cash | 354,170 | 35.42% |
| No Charge | 5,503 | 0.55% |
| Dispute | 1,751 | 0.18% |

**`passenger_count`: Passengers per Trip**

| Count | Trips | % |
|---|---|---|
| 1 | 702,047 | 70.20% |
| 2 | 149,485 | 14.95% |
| 5 | 47,101 | 4.71% |
| 3 | 46,451 | 4.65% |
| 6 | 30,817 | 3.08% |
| 4 | 24,003 | 2.40% |
| 0 | 88 | 0.01% |
| 7–9 | 8 | 0.00% |

**`store_and_fwd_flag`: Store and Forward Flag**

| Flag | Count | % |
|---|---|---|
| N (not stored) | 997,324 | 99.73% |
| Y (stored) | 2,676 | 0.27% |

---

### Plot Observations

**`vendorid`**: Reasonably balanced split between the two vendors. VeriFone holds a slight majority (56.19%) over Creative Mobile (43.81%). No cause for concern, both vendors are well represented in the dataset. Will be one-hot encoded in Phase 4.

**`ratecodeid`**: Extremely dominant Standard rate (96.73%) dwarfs all other codes visually. JFK is the only meaningful minority class at 2.56%, confirming that airport trips are the primary non-standard fare type. The 15 invalid code-99 rows are confirmed for removal in Phase 3. Non-standard codes (JFK, Newark, Nassau) likely correspond to flat-rate fares that could distort a distance-based regression, will be one-hot encoded as binary flags in Phase 4.

**`payment_type`**: Near-binary split between Credit Card (63.86%) and Cash (35.42%). No Charge and Dispute together account for less than 1% and will be removed in Phase 3 as they do not represent standard fare transactions. Notably, `tip_amount` is only recorded for credit card payments, using tip as a feature would introduce payment-type leakage into the model.

**`passenger_count`**: Solo riders dominate at 70.20%, with pairs at 14.95% being a distant second. An interesting anomaly: groups of 5
(4.71%) appear more frequently than groups of 3 (4.65%) and 4 (2.40%), suggesting minivan or shared rides. The 88 zero-passenger trips are confirmed invalid and flagged for removal in Phase 3. Counts of 7, 8, and 9 are negligible (8 total trips). Will be binned into three groups in Phase 4: solo (1), small group (2–3), large group (4+).

**`store_and_fwd_flag`**: Near-constant at N (99.73%) with only 2,676 stored-forward trips (0.27%). This column has almost no variance and therefore no predictive signal. Will be dropped in Phase 4.

---

### Encoding Strategy Confirmed for Phase 4

| Column | Strategy | Reason |
|---|---|---|
| `vendorid` | One-hot encode | True binary categorical |
| `ratecodeid` | One-hot encode | Codes have no ordinal meaning |
| `payment_type` | Binary flag (credit card vs other) | Near-binary, rest < 1% |
| `passenger_count` | Bin → solo / small / large group | Multimodal, not continuous |
| `store_and_fwd_flag` | Drop | 99.73% constant, no signal |

In [ ]:
# ------------------------------------------------------------
# Step 2.4: Temporal Patterns
# ------------------------------------------------------------

def extract_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract temporal components from the pickup datetime column.

    Creates the following columns on a copy of the DataFrame:
      - pickup_hour: hour of day (0–23)
      - pickup_day: day of week (0=Monday, ..., 6=Sunday)
      - pickup_day_name: full day name (Monday, ..., Sunday)
      - pickup_date: date only (no time component)

    Parameters
    ----------
    df: pd.DataFrame
        Raw DataFrame containing ``tpep_pickup_datetime``.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with temporal columns appended.
    """
    df = df.copy()
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_day"] = df["tpep_pickup_datetime"].dt.dayofweek
    df["pickup_day_name"] = df["tpep_pickup_datetime"].dt.day_name()
    df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
    return df

def plot_temporal_patterns(df: pd.DataFrame) -> None:
    """
    Produce four temporal analysis plots:
      1. Trip volume by hour of day
      2. Average fare by hour of day
      3. Trip volume by day of week
      4. Trip volume by date (daily trend)

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with temporal columns added by
        ``extract_temporal_features()``.
    """
    # Aggregations
    hourly_volume = (
        df.groupby("pickup_hour")
        .size()
        .reset_index(name="trip_count")
    )
    hourly_fare = (
        df.groupby("pickup_hour")["fare_amount"]
        .mean()
        .round(2)
        .reset_index(name="avg_fare")
    )

    day_order  = ["Friday", "Saturday", "Sunday", "Monday", "Tuesday"]
    daily_vol  = (
        df.groupby("pickup_day_name")
        .size()
        .reindex(day_order)
        .reset_index(name="trip_count")
    )
    daily_vol.columns = ["day_name", "trip_count"]

    date_vol = (
        df.groupby("pickup_date")
        .size()
        .reset_index(name="trip_count")
    )

    # Plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("Temporal Patterns, Trip Volume & Fare", fontsize=15)

    # Plot 1: Trip volume by hour
    axes[0, 0].bar(
        hourly_volume["pickup_hour"],
        hourly_volume["trip_count"],
        color=PALETTE["primary"], edgecolor="white", linewidth=0.3,
    )
    axes[0, 0].set_title("Trip Volume by Hour of Day")
    axes[0, 0].set_xlabel("Hour of Day")
    axes[0, 0].set_ylabel("Number of Trips")
    axes[0, 0].set_xticks(range(0, 24))
    axes[0, 0].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1_000:.0f}K")
    )

    # Plot 2: Average fare by hour
    axes[0, 1].plot(
        hourly_fare["pickup_hour"],
        hourly_fare["avg_fare"],
        color=PALETTE["secondary"], linewidth=2, marker="o", markersize=4,
    )
    axes[0, 1].fill_between(
        hourly_fare["pickup_hour"],
        hourly_fare["avg_fare"],
        alpha=0.15, color=PALETTE["secondary"],
    )
    axes[0, 1].set_title("Average Fare by Hour of Day")
    axes[0, 1].set_xlabel("Hour of Day")
    axes[0, 1].set_ylabel("Average Fare ($)")
    axes[0, 1].set_xticks(range(0, 24))

    # Plot 3: Trip volume by day of week
    axes[1, 0].bar(
        daily_vol["day_name"],
        daily_vol["trip_count"],
        color=PALETTE["accent"], edgecolor="white", linewidth=0.3,
    )
    axes[1, 0].set_title("Trip Volume by Day of Week")
    axes[1, 0].set_xlabel("Day")
    axes[1, 0].set_ylabel("Number of Trips")
    axes[1, 0].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1_000:.0f}K")
    )

    # Plot 4: Daily trip trend
    axes[1, 1].plot(
        date_vol["pickup_date"].astype(str),
        date_vol["trip_count"],
        color=PALETTE["neutral"], linewidth=2,
        marker="o", markersize=6,
    )
    axes[1, 1].fill_between(
        date_vol["pickup_date"].astype(str),
        date_vol["trip_count"],
        alpha=0.15, color=PALETTE["neutral"],
    )
    axes[1, 1].set_title("Daily Trip Volume Trend")
    axes[1, 1].set_xlabel("Date")
    axes[1, 1].set_ylabel("Number of Trips")
    axes[1, 1].tick_params(axis="x", rotation=25)
    axes[1, 1].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x/1_000:.0f}K")
    )

    plt.tight_layout()
    save_figure(fig, "phase2_temporal_patterns")
    plt.show()

# Run
df_temporal = extract_temporal_features(df_raw)

# SQL summary for quick reference
print("\n" + "=" * 70)
print("Hourly Volume & Fare Summary (SQL)")
print("=" * 70)

hourly_summary = quick_sql("""
    SELECT
        HOUR(tpep_pickup_datetime) AS hour,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(MIN(fare_amount), 2) AS min_fare,
        ROUND(MAX(fare_amount), 2) as max_fare
    FROM trips
    GROUP BY 1
    ORDER BY 1
""")
print(hourly_summary.to_string(index=False))

print("\n" + "=" * 70)
print("Daily Volume Summary (SQL)")
print("=" * 70)
daily_summary = quick_sql("""
    SELECT
        DAYNAME(tpep_pickup_datetime) AS day_name,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM trips
    GROUP BY 1
    ORDER BY trip_count DESC
""")

print(daily_summary.to_string(index=False))

plot_temporal_patterns(df_temporal)


## Step 2.4: Temporal Patterns ✅

### Hourly Volume & Fare Summary

| Hour | Trip Count | Avg Fare | Observation |
|---|---|---|---|
| 00:00 | 36,471 | \$13.27 | Late night: moderate volume |
| 04:00 | 12,208 | \$14.71 | Overnight trough begins |
| 05:00 | 10,701 | \$16.50 | **Lowest volume, highest avg fare** |
| 06:00 | 21,755 | \$13.59 | Morning ramp-up begins |
| 08:00 | 43,948 | \$12.52 | Morning commute peak |
| 14:00 | 53,679 | \$14.04 | Afternoon build |
| 18:00 | 60,660 | \$12.70 | **Peak volume, lower avg fare** |
| 19:00 | 59,253 | \$12.18 | **Lowest avg fare of the day** |
| 23:00 | 44,361 | \$13.72 | Late night premium |

### Daily Volume Summary

| Day | Trip Count | Avg Fare | Note |
|---|---|---|---|
| Saturday | 254,633 | \$12.66 | Busiest day |
| Monday | 241,900 | \$13.13 | Strong weekday demand |
| Sunday | 227,861 | \$13.82 | Highest avg fare |
| Friday | 226,872 | \$13.16 | Consistent with weekday |
| Tuesday | 48,734 | \$12.59 | ⚠️ Partial day, data ends 08:59 |

---

### Plot Observations

**Plot 1: Trip Volume by Hour of Day:**
- Clear **bimodal pattern**: a small early-morning cluster (0–1am) representing late-night rides, followed by a trough at 4–5am, then
  a sustained build from 6am through an evening peak at 18:00 (60,660 trips)
- The evening rush (17:00–20:00) is the busiest period of the day, likely driven by commuters and leisure trips combined
- The pre-dawn trough (4–5am) records the lowest volume (~10–12K trips per hour), consistent with NYC nightlife winding down before the
  morning commute begins

**Plot 2: Average Fare by Hour of Day:**
- A notable **inverse relationship** between volume and fare: the highest average fare (\\$16.50) occurs at 5am when volume is lowest,
  while the lowest average fare (\\$12.18) occurs at 7pm when volume peaks
- This is explained by trip composition: early morning rides are predominantly longer airport and outer-borough trips commanding
  higher fares, while evening peak-hour trips are dominated by short intra-Manhattan commutes
- A secondary fare elevation at 23:00 (\$13.72) suggests late-night trips also tend to be longer or to outer boroughs
- The fare curve is remarkably stable (\$12–\$14) through the 8am–4pm window, suggesting consistent mid-day trip length distributions

**Plot 3: Trip Volume by Day of Week:**
- Saturday is the busiest day (254,633 trips) followed closely by Monday (241,900) and Sunday (227,861)
- All four complete days show volumes within a tight 226K–255K band, confirming consistent demand across the sample window
- Tuesday appears anomalously low (48,734), this is **not a true signal** but a data truncation artefact: the sample ends at 2017-08-15 08:59, capturing only the first 9 hours of Tuesday

**Plot 4: Daily Trip Volume Trend:**
- The sharp drop on August 15 (Tuesday) visually confirms the truncation artefact, the line falls from ~245K to ~49K
  representing a partial day, not a real demand collapse, the four complete days (Aug 11–14) show a stable trend with a
  Saturday peak, consistent with expected NYC weekend demand

---

### Key Feature Engineering Implications for Phase 4

| Feature | Derivation | Rationale |
|---|---|---|
| `pickup_hour` | `dt.hour` | Strong volume and fare signal |
| `is_rush_hour` | hours 7–9 and 17–20 = 1 | Captures commute demand patterns |
| `is_overnight` | hours 0–5 = 1 | Higher avg fare, longer trips |
| `pickup_day_of_week` | `dt.dayofweek` | Weekend vs weekday demand shift |
| `is_weekend` | Sat/Sun = 1 | Saturday is peak demand day |

> ⚠️ **Important note for modelling:** Tuesday data is a partial day
> (9 hours only). The `pickup_date` column should **not** be used as
> a raw feature, temporal signals should be extracted as cyclic or
> binary engineered features to avoid the model learning date-specific
> artefacts from this limited 4-day window.

## Step 2.5: Correlation Analysis
Measure linear relationships between all numeric features and the target variable `fare_amount`. This identifies the strongest predictors and flags multicollinearity between features, both critical inputs for feature selection in Phase 4.

In [ ]:
# ------------------------------------------------------------
# Step 2.5: Correlation Analysis
# ------------------------------------------------------------

def compute_correlation_matrix(
    df:pd.DataFrame,
    exclude: list[str] = None
) -> pd.DataFrame:
    """
    Compute the Pearson correlation matrix for all numeric columns.

    Non-numeric columns and explicitly excluded columns are dropped
    before computation. Object-typed columns that are coercible to
    numeric (e.g. ``extra``, ``total_amount``) are cast automatically.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to analyze.
    exclude: list[str], optional
        Columns to exclude from the matrix (e.g. ID columns).

    Returns
    -------
    pd.DataFrame
        Square correlation matrix of shape (n_features x n_features).
    """

    exclude = exclude or []

    # Coerce object columns that should be numeric
    df_num = df.copy()
    for col in df_num.select_dtypes(include="object").columns:
        df_num[col] = pd.to_numeric(df_num[col], errors="coerce")

    # Keep only numeric columns, drop exclusions
    df_num = df_num.select_dtypes(include=[np.number])
    df_num = df_num.drop(columns=[c for c in exclude if c in df_num.columns])

    return df_num.corr(method="pearson")

def compute_target_correlations(
    corr_matrix: pd.DataFrame,
    target_col: str = TARGET_COL
) -> pd.Series:
    """
    Extract and sort correlations between all features and the target.

    Parameters
    ----------
    corr_matrix : pd.DataFrame
        Full correlation matrix from ``compute_correlation_matrix()``.
    target_col  : str
        Name of the target column.

    Returns
    -------
    pd.Series
        Correlations with the target, sorted by absolute value descending,
        excluding the target's self-correlation.
    """
    target_corr = corr_matrix[target_col].drop(labels=[target_col])
    return target_corr.reindex(
        target_corr.abs().sort_values(ascending=False).index
    )

def plot_correlation_analysis(
    corr_matrix  : pd.DataFrame,
    target_corr  : pd.Series,
    target_col   : str = TARGET_COL,
) -> None:
    """
    Produce two correlation visualisations:
      1. Full correlation heatmap across all numeric features
      2. Horizontal bar chart of per-feature correlation with target

    Parameters
    ----------
    corr_matrix : pd.DataFrame
        Full Pearson correlation matrix.
    target_corr : pd.Series
        Per-feature correlations with the target variable.
    target_col  : str
        Name of the target column (used for plot labels).
    """
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Correlation Analysis", fontsize=15)

    # Plot 1: Full heatmap
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask = mask,
        ax = axes[0],
        annot = True,
        fmt = ".2f",
        cmap = "coolwarm",
        center = 0,
        linewidths = 0.5,
        linecolor = "white",
        cbar_kws = {"shrink": 0.8},
        annot_kws = {"size": 8}
    )
    axes[0].set_title("Pearson Correlation Heatmap\n(lower triangle)")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].tick_params(axis="y", rotation=0)

    # Plot 2: Target correlations bar chart
    colors = [
        PALETTE["primary"] if v >= 0 else PALETTE["accent"]
        for v in target_corr.values
    ]
    bars = axes[1].barh(
        target_corr.index,
        target_corr.values,
        color=colors,
        edgecolor="white",
        linewidth=0.4
    )
    axes[1].axvline(0, color="#374151", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Feature Correlation with `{target_col}`\n(sorted by |r|)")
    axes[1].set_xlabel("Pearson r")
    axes[1].invert_yaxis()

    # Annotate bars with correlation values
    for bar, val in zip(bars, target_corr.values):
        axes[1].text(
            val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            ha="left" if val >= 0 else "right",
            fontsize=9
        )

    plt.tight_layout()
    save_figure(fig, "phase2_correlation_analysis")
    plt.show()

# Run
EXCLUDE_FROM_CORR = ["id", "pulocationid", "dolocationid"]

corr_matrix = compute_correlation_matrix(df_raw, exclude=EXCLUDE_FROM_CORR)
target_corr = compute_target_correlations(corr_matrix, target_col=TARGET_COL)

print("\n" + "=" * 70)
print("Feature Correlations with fare_amount")
print("=" * 70)
print(f"\n  {'Feature':<25} {'Pearson r':>10} {'Strength'}")
print(f"  {'-' * 50}")

for feat, val in target_corr.items():
    strength = (
        "Strong" if abs(val) >= 0.5 else
        "Moderate" if abs(val) >= 0.3 else
        "Weak" if abs(val) >= 0.1 else
        "Negligible"
    )
    direction = "positive" if val > 0 else "negative"
    print(f"  {feat:<25} {val:>10.4f} {strength} {direction}")

print("\n" + "=" * 70)
print("Multicollinearity Check")
print("=" * 70)
# Flag feature pairs with |r| > 0.7 (excluding target)
feat_corr = corr_matrix.drop(
    columns=[TARGET_COL], index=[TARGET_COL]
)
high_corr_pairs = []
for i in range(len(feat_corr.columns)):
    for j in range(i + 1, len(feat_corr.columns)):
        r = feat_corr.iloc[i, j]
        if abs(r) >= 0.7:
            high_corr_pairs.append({
                "feature_1": feat_corr.columns[i],
                "feature_2": feat_corr.columns[j],
                "pearson_r": round(r, 4)
            })

if high_corr_pairs:
    print(f"\n  ⚠️  {len(high_corr_pairs)} highly correlated feature pair(s) found:")
    for pair in high_corr_pairs:
        print(f"  {pair['feature_1']:<25} <-> {pair['feature_2']:<25} r = {pair['pearson_r']}")
else:
    print("\n  ✅ No highly correlated feature pairs found (threshold |r| >= 0.7)")

plot_correlation_analysis(corr_matrix, target_corr)


## Step 2.5: Correlation Analysis ✅

### Feature Correlations with `fare_amount`

| Feature | Pearson r | Strength | Note |
|---|---|---|---|
| `total_amount` | 0.982 | Strong positive | ⚠️ Data leakage, derived from fare |
| `trip_distance` | 0.900 | Strong positive | ✅ Strongest legitimate predictor |
| `tolls_amount` | 0.585 | Strong positive | ✅ Longer trips incur more tolls |
| `tip_amount` | 0.557 | Strong positive | ⚠️ Post-ride value, potential leakage |
| `ratecodeid` | 0.355 | Moderate positive | ✅ Airport/special rates drive higher fares |
| `mta_tax` | -0.208 | Weak negative | ⚠️ Near-constant, correlation unreliable |
| `extra` | 0.052 | Negligible | Low signal |
| `payment_type` | -0.051 | Negligible | Low signal |
| `improvement_surcharge` | 0.041 | Negligible | Near-constant, no signal |
| `vendorid` | 0.019 | Negligible | No meaningful fare difference by vendor |
| `passenger_count` | 0.016 | Negligible | Passenger count does not drive fare |
| `store_and_fwd_flag` | NaN | N/A | ⚠️ Object dtype, excluded from correlation |

### Multicollinearity Check

| Feature 1 | Feature 2 | Pearson r | Action |
|---|---|---|---|
| `trip_distance` | `total_amount` | 0.899 | Moot, `total_amount` dropped as leakage |

---

### Plot Observations

**Heatmap (lower triangle):**
- The dominant red cell is `trip_distance` ↔ `fare_amount` (r=0.90) and `total_amount` ↔ `fare_amount` (r=0.98), both clearly visible as the darkest cells in the fare row
- `tip_amount` ↔ `total_amount` (r=0.68) and `tolls_amount` <-> `total_amount` (r=0.66) are strongly coloured, expected since `total_amount` is a sum of all charge components
- `tip_amount` ↔ `payment_type` shows a notable negative correlation (r=-0.49), confirming that cash payments (type 2) systematically record zero tips since cash tips are not metered
- `mta_tax` ↔ `ratecodeid` (r=-0.37) reflects that special rate codes (JFK, Newark) bypass the standard MTA tax trigger
- Most feature pairs show negligible correlations (near-white cells), suggesting low multicollinearity risk among the legitimate features

**Target Correlation Bar Chart:**
- `total_amount` bar (r=0.982) towers over all others, a clear visual signal of leakage that must be removed before modelling
- `trip_distance` (r=0.900) is the dominant legitimate predictor by a wide margin, the model will lean heavily on this feature
- `mta_tax` is the only feature with a meaningful negative bar (r=-0.208), driven by its near-constant nature rather than a true inverse relationship with fare
- `store_and_fwd_flag` returns NaN correlation because it is stored as an object dtype (Y/N strings), it will be encoded or dropped in Phase 4

---

### Critical Leakage Flags 🚨

| Column | Issue | Action in Phase 4 |
|---|---|---|
| `total_amount` | Computed directly from `fare_amount`, r=0.982 | **DROP**: hard leakage |
| `tip_amount` | Recorded after ride completion, not available at pickup | **DROP**: temporal leakage |
| `improvement_surcharge` | Near-constant \\$0.30, no predictive signal | **DROP** |
| `mta_tax` | Near-constant \\$0.50, correlation unreliable | **DROP** |
| `store_and_fwd_flag` | Object dtype, 99.73% constant, NaN correlation | **DROP** |

### Confirmed Predictive Features for Modelling

| Feature | Pearson r | Phase 4 Treatment |
|---|---|---|
| `trip_distance` | 0.900 | log1p transform |
| `tolls_amount` | 0.585 | log1p transform |
| `ratecodeid` | 0.355 | One-hot encode |
| `payment_type` | -0.051 | Binary flag |
| `vendorid` | 0.019 | One-hot encode |
| `passenger_count` | 0.016 | Bin into groups |
| `extra` | 0.052 | Keep as-is |
| `pickup_hour` | - | Engineer from datetime |
| `is_rush_hour` | - | Engineer from datetime |

## Step 2.6: Fare Drivers

Visualise the relationship between the strongest legitimate predictors and `fare_amount`. This confirms which features drive fare variation and how, linear non-linear, or categorical step-changes, which directly informs model selection and feature transformation decisions in Phase 4.

In [ ]:
# ------------------------------------------------------------
# Step 2.6: Fare Drivers
# ------------------------------------------------------------

def plot_continuous_fare_drivers(df: pd.DataFrame) -> None:
    """
    Plot scatter relationships between continuous features and fare_amount.

    Each subplot shows a hexbin density plot (preferred over scatter for
    1M rows) with a LOWESS trend line to reveal the underlying relationship
    shape, linear vs non-linear.

    Parameters
    ----------
    df : pd.DataFrame
        The raw DataFrame containing fare and feature columns.
    """
    # Clip to p1-p99 for readability
    df_plot = df.copy()
    for col in ["fare_amount", "trip_distance", "tolls_amount"]:
        p1 = df_plot[col].quantile(0.01)
        p99 = df_plot[col].quantile(0.99)
        df_plot[col] = df_plot[col].clip(lower=p1, upper=p99)

    continuous_drivers = [
        ("trip_distance", "Trip Distance (miles)"),
        ("tolls_amount", "Tolls Amount ($)")
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Continuous Feature vs fare_amount (hexbin density)", fontsize=15)

    for ax, (col, label) in zip(axes, continuous_drivers):
        hb = ax.hexbin(
            df_plot[col],
            df_plot["fare_amount"],
            gridsize = 60,
            cmap = "Blues",
            mincnt = 1
        )
        plt.colorbar(hb, ax=ax, label="Trip Count")
        ax.set_xlabel(label)
        ax.set_ylabel("Fare Amount ($)")
        ax.set_title(f"{label} vs Fare Amount")

    plt.tight_layout()
    save_figure(fig, "phase2_continuous_fare_drivers")
    plt.show()

def plot_categorical_fare_drivers(df: pd.DataFrame) -> None:
    """
    Plot average fare amount broken down by categorical features using
    box plots and bar charts.

    Features analysed:
      - ``ratecodeid`` : flat-rate vs metered fares
      - ``passenger_count`` : group size effect
      - ``payment_type`` : payment method effect
      - ``vendorid`` : vendor effect

    Parameters
    ----------
    df : pd.DataFrame
        The raw DataFrame.
    """
    # Clip fare to p1-p99 for readability
    p1  = df["fare_amount"].quantile(0.01)
    p99 = df["fare_amount"].quantile(0.99)
    df_plot = df.copy()
    df_plot["fare_amount"] = df_plot["fare_amount"].clip(lower=p1, upper=p99)

    # Apply readable labels
    df_plot["rate_label"] = df_plot["ratecodeid"].map({
        1: "Standard", 2: "JFK", 3: "Newark",
        4: "Nassau/WC", 5: "Negotiated", 6: "Group", 99: "Invalid"
    })
    df_plot["payment_label"] = df_plot["payment_type"].map({
        1: "Credit Card", 2: "Cash", 3: "No Charge", 4: "Dispute"
    })
    df_plot["vendor_label"] = df_plot["vendorid"].map({
        1: "Creative Mobile", 2: "VeriFone"
    })

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Categorical Feature vs fare_amount", fontsize=15)

    # Plot 1: Rate code box plot
    rate_order = ["Standard", "JFK", "Newark", "Nassau/WC", "Negotiated", "Group"]
    sns.boxplot(
        data = df_plot[df_plot["rate_label"].isin(rate_order)],
        x = "rate_label",
        y = "fare_amount",
        order = rate_order,
        ax = axes[0, 0],
        palette = "Blues",
        linewidth = 0.8
    )
    axes[0, 0].set_title("Fare by Rate Code")
    axes[0, 0].set_xlabel("Rate Code")
    axes[0, 0].set_ylabel("Fare Amount ($)")
    axes[0, 0].tick_params(axis="x", rotation=20)

    # Plot 2: Passenger count avg fare bar
    pax_fare = (
        df_plot[df_plot["passenger_count"].between(1, 6)]
        .groupby("passenger_count")["fare_amount"]
        .mean()
        .round(2)
        .reset_index()
    )
    axes[0, 1].bar(
        pax_fare["passenger_count"].astype(str),
        pax_fare["fare_amount"],
        color=PALETTE["primary"], edgecolor="white", linewidth=0.4
    )
    for i, row in pax_fare.iterrows():
        axes[0, 1].text(
            i, row["fare_amount"] + 0.1,
            f"${row['fare_amount']:.2f}",
            ha="center", fontsize=9
        )
    axes[0, 1].set_title("Average Fare by Passenger Count")
    axes[0, 1].set_xlabel("Passenger Count")
    axes[0, 1].set_ylabel("Average Fare ($)")
    axes[0, 1].set_ylim(0, pax_fare["fare_amount"].max() * 1.5)

    # Plot 3: Payment type box plot
    pay_order = ["Credit Card", "Cash", "No Charge", "Dispute"]
    sns.boxplot(
        data = df_plot[df_plot["payment_label"].isin(pay_order)],
        x = "payment_label",
        y = "fare_amount",
        order = pay_order,
        ax = axes[1, 0],
        palette = "Greens",
        linewidth = 0.8
    )
    axes[1, 0].set_title("Fare by Payment Type")
    axes[1, 0].set_xlabel("Payment Type")
    axes[1, 0].set_ylabel("Fare Amount ($)")

    # Plot 4: Vendor box plot
    sns.boxplot(
        data = df_plot,
        x = "vendor_label",
        y = "fare_amount",
        ax = axes[1, 1],
        palette = "Oranges",
        linewidth = 0.8
    )
    axes[1, 1].set_title("Fare by Vendor")
    axes[1, 1].set_xlabel("Vendor")
    axes[1, 1].set_ylabel("Fare Amount ($)")

    plt.tight_layout()
    save_figure(fig, "phase2_categorical_fare_drivers")
    plt.show()

def summarize_fare_by_category(df: pd.DataFrame) -> None:
    """
    Print SQL-powered average fare summaries foir key categorical features.

    Parameters
    ----------
    df:pd.DataFrame
        The raw DataFrame (DuckDB table ``trips`` must be registered).
    """
    queries = {
        "Rate Code": """
            SELECT
                ratecodeid,
                COUNT(*) AS trip_count,
                ROUND(AVG(fare_amount), 2) AS avg_fare,
                ROUND(MIN(fare_amount), 2) AS min_fare,
                ROUND(MAX(fare_amount), 2) AS max_fare
            FROM trips
            WHERE ratecodeid BETWEEN 1 AND 6
            GROUP BY 1 ORDER BY avg_fare DESC
        """,
        "Passenger Count": """
            SELECT
                passenger_count,
                COUNT(*) AS trip_count,
                ROUND(AVG(fare_amount), 2) AS avg_fare
            FROM trips
            WHERE passenger_count BETWEEN 1 AND 6
            GROUP BY 1 ORDER BY 1
        """,
        "Payment Type": """
            SELECT
                payment_type,
                COUNT(*) AS trip_count,
                ROUND(AVG(fare_amount), 2) AS avg_fare
            FROM trips
            GROUP BY 1 ORDER BY avg_fare DESC
        """
    }

    for title, query in queries.items():
        print(f"\n{'-' * (35 - len(title))}  Avg Fare by {title} {'-' * (35 - len(title))}")
        print(quick_sql(query).to_string(index=False))

# Run
summarize_fare_by_category(df_raw)
plot_continuous_fare_drivers(df_raw)
plot_categorical_fare_drivers(df_raw)


## Step 2.6: Fare Drivers - relationship between key features and fare_amount ✅

### SQL Summary Tables

**Average Fare by Rate Code**

| Rate Code | Trips | Avg Fare | Min | Max | Observation |
|---|---|---|---|---|---|
| Newark (3) | 2,456 | \\$66.17 | -\\$22.00 | \\$216.50 | Flat rate, outer borough |
| Nassau/WC (4) | 670 | \\$66.04 | \\$0.00 | \\$621.50 | Flat rate, outer borough |
| Negotiated (5) | 3,981 | \\$57.60 | -\\$233.00 | \\$600.00 | Wide range, driver negotiated |
| JFK (2) | 25,555 | \\$51.84 | -\\$52.00 | \\$52.00 | Fixed flat rate, \\$52.00 |
| Standard (1) | 967,313 | \\$11.77 | -\\$37.50 | \\$414.50 | Metered, vast majority |
| Group (6) | 10 | \\$3.50 | \\$2.50 | \\$5.50 | Negligible, only 10 trips |

**Average Fare by Passenger Count**

| Passengers | Trips | Avg Fare | Observation |
|---|---|---|---|
| 1 | 702,047 | $12.93 | Baseline |
| 2 | 149,485 | $13.75 | Slight increase |
| 3 | 46,451 | $13.91 | Marginal increase |
| 4 | 24,003 | $14.60 | Slight peak |
| 5 | 47,101 | $13.11 | Drops back |
| 6 | 30,817 | $13.07 | Near baseline |

**Average Fare by Payment Type**

| Payment Type | Trips | Avg Fare | Observation |
|---|---|---|---|
| Credit Card (1) | 638,576 | \\$13.62 | Slightly higher avg |
| Cash (2) | 354,170 | \\$12.33 | Slightly lower avg |
| Dispute (4) | 1,751 | \\$12.19 | Small sample |
| No Charge (3) | 5,503 | \\$12.06 | Comp/waived fares |

---

### Plot Observations

**Image 1: Continuous Fare Drivers (hexbin density):**

**`trip_distance` vs `fare_amount`:**
- The hexbin reveals a **strongly linear relationship** in the dense core cluster (0–5 miles, \\$0–\\$20), the darkest hexagons trace a clear diagonal band confirming trip distance is the primary metered fare driver
- A striking **horizontal band at exactly \\$52** is visible at the top of the plot extending across multiple distance values, these are JFK flat-rate trips (\\$52 fixed regardless of distance), confirming `ratecodeid` introduces step-changes that a purely linear model will struggle to capture without rate code flags
- Beyond 5 miles the relationship remains linear but density drops sharply, with sparse long-distance trips scattered across the upper right, these are the airport and outer-borough runs
- The log1p transform on `trip_distance` will compress the right tail and improve model fit on the dense short-trip cluster

**`tolls_amount` vs `fare_amount`:**
- A massive **vertical spike at \\$0** dominates the left side, confirming that the overwhelming majority of trips incur no tolls
- Two secondary clusters are visible: one at ~\\$3.17 (single bridge or tunnel crossing) and one at ~\\$6.12 (double crossing or higher toll), both showing a vertical spread of fares, meaning toll amount alone does not determine fare
- The correlation (r=0.585) is driven by the fact that longer trips to outer boroughs and airports both incur higher fares **and** tolls, it is a proxy for trip length rather than a direct fare driver
- The rightmost cluster at ~\\$6.50 corresponds to the maximum p99 clip point

**Image 2: Categorical Fare Drivers:**

**Fare by Rate Code (box plot):**
- The most visually striking finding: **JFK (code 2) shows an extremely tight box** with nearly zero IQR, the whiskers and box collapse to a near-horizontal line at \\$52, confirming it is a fixed flat-rate fare with almost no variance
- **Newark (3) and Nassau/WC (4)** show high medians (~\\$40–50) with wide boxes, flat rates but with more variance than JFK, likely
  driven by negotiated surcharges and route variations
- **Negotiated (5)** has the widest spread of all codes, by definition these fares are driver-agreed and unconstrained, ranging
  from near-zero to \\$600
- **Standard (1)** has the lowest median (~\\$9–10) and tightest distribution after JFK, consistent with short metered city trips
- **Group (6)** with only 10 trips shows an anomalously low median ($3.50), too few samples to be meaningful

**Average Fare by Passenger Count (bar chart):**
- Remarkably flat across all group sizes, fares range only from \\$12.77 (solo) to \\$14.28 (4 passengers), a spread of just \\$1.51
- This visually confirms the near-zero Pearson r (0.016), passenger count has virtually no effect on fare amount
- The slight peak at 4 passengers may reflect larger vehicles or outer-borough group trips rather than a true passenger-count effect
- Confirms the decision to either bin or drop this feature in Phase 4

**Fare by Payment Type (box plot):**
- Near-identical box shapes between Credit Card and Cash, the difference in avg fare (\\$13.62 vs \\$12.33) is too small to represent
  a meaningful fare driver
- Dispute category shows a higher upper whisker, suggesting disputed fares tend to be on longer/more expensive trips
- Confirms payment type is not a direct fare driver, any observed difference is likely a confound with trip type and tip recording

**Fare by Vendor (box plot):**
- Virtually indistinguishable distributions between VeriFone and Creative Mobile, median, IQR, and whisker lengths are nearly
  identical for both vendors
- Confirms r=0.019, vendor identity has no meaningful effect on fare
- Both vendors operate under identical TLC metered rate structures

---

### Key Fare Driver Summary

| Feature | Relationship Type | Strength | Phase 4 Action |
|---|---|---|---|
| `trip_distance` | Linear with flat-rate step-changes | Very strong (r=0.90) | log1p transform + keep |
| `ratecodeid` | Step-change, each code a distinct regime | Moderate (r=0.355) | One-hot encode |
| `tolls_amount` | Proxy for trip length, zero-inflated | Moderate (r=0.585) | log1p transform + keep |
| `passenger_count` | Negligible, flat across all group sizes | Negligible (r=0.016) | Bin or drop |
| `payment_type` | No meaningful fare effect | Negligible (r=-0.051) | Binary flag or drop |
| `vendorid` | No meaningful fare effect | Negligible (r=0.019) | Drop |

> ⚠️ **JFK flat-rate insight:** The fixed \\$52 JFK fare creates a
> horizontal band in the `trip_distance` vs `fare_amount` plot that
> violates the linear distance-fare assumption. One-hot encoding
> `ratecodeid` will allow the model to learn separate intercepts for
> each rate regime, significantly improving prediction accuracy for
> non-standard rate trips.

## Step 2.7: Outlier Profiling
Quantify and characterise outliers across key numeric features using the IQR method. Understanding outlier volume, direction, and business context determines whether they should be dropped, capped, or flagged in Phase 3.

In [ ]:
# ------------------------------------------------------------
# Step 2.7: Outlier Profiling
# ------------------------------------------------------------

def compute_iqr_outliers(
    df: pd.DataFrame,
    cols: list[str],
    k: float = 1.5
) -> pd.DataFrame:
    """
    Detect outliers using the Tukey IQR fence method.

    A value is flagged as an outlier if it falls below 
    Q1 - k*IQR or above Q3 + k*IQR.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to analyze.
    cols: list[str]
        Numeric columns to profile.
    k: float, optional
        IQR multiplier for fence width (default 1.5 - standard Tukey)

    Returns
    -------
    pd.DataFrame
        Outlier summary with Q1, Q3, IQR, fences and counts per column,
        sorted by total outlier count descending.
    """
    records = []
    for col in cols:
        series = pd.to_numeric(df[col], errors="coerce").dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - (k * iqr)
        upper = q3 + (k * iqr)

        n_lower = int((series < lower).sum())
        n_upper = int((series > upper).sum())
        n_total = n_lower + n_upper

        records.append({
            "column": col,
            "q1": round(q1, 2),
            "q3": round(q3, 2),
            "iqr": round(iqr, 2),
            "lower_fence": round(lower, 2),
            "upper_fence": round(upper, 2),
            "below_fence": n_lower,
            "above_fence": n_upper,
            "total_outliers": n_total,
            "outlier_pct": round(n_total / len(df) * 100, 4)
        })

    return (
        pd.DataFrame(records)
        .sort_values("total_outliers", ascending=False)
        .reset_index(drop=True)
    )

def plot_outlier_boxplots(
    df: pd.DataFrame,
    cols: list[str]
) -> None:
    """
    Plot box plots for each numeric feature to visually identify outlier spread,
    fence positions and distribution symmetry.

    Each subplot clips display to p1-p99 to keep the boxes readable while still 
    showing the outlier dots beyond the whiskers.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to plot.
    cols: list[str]
        Numeric columns to include
    """
    n_cols = 3
    n_rows = int(np.ceil(len(cols) / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 6, n_rows * 4)
    )
    axes = axes.flatten()

    for i, col in enumerate(cols):
        series = pd.to_numeric(df[col], errors="coerce").dropna()
        p1, p99 = series.quantile(0.01), series.quantile(0.99)
        clipped = series.clip(lower=p1, upper=p99)

        axes[i].boxplot(
            clipped,
            vert = True,
            patch_artist = True,
            boxprops = dict(facecolor=PALETTE["primary"], alpha=0.6),
            medianprops = dict(color=PALETTE["accent"], linewidth=2),
            whiskerprops = dict(color=PALETTE["neutral"]),
            capprops = dict(color=PALETTE["neutral"]),
            flierprops = dict(
                marker = "o",
                markerfacecolor = PALETTE["accent"],
                markersize = 2,
                alpha = 0.3,
                linestyle = "none"
            )
        )
        axes[i].set_title(col)
        axes[i].set_ylabel("Value")
        axes[i].set_xticks([])

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Outlier Profiles: Box Plots (clipped p1-p99)", fontsize=15)
    plt.tight_layout()
    save_figure(fig, "phase2_outlier_boxplots")
    plt.show()

def plot_outlier_summary(outlier_df: pd.DataFrame) -> None:
    """
    Plot a horizontal bar chart summarizing outlier counts per column.

    Parameters
    ----------
    outlier_df: pd.DataFrame
        Output of ``compute_iqr_outliers()``.
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    bars = ax.barh(
        outlier_df["column"],
        outlier_df["total_outliers"],
        color=PALETTE["accent"],
        edgecolor="white",
        linewidth=0.4
    )
    ax.invert_yaxis()
    ax.set_xlabel("Number of Outliers (IQR method)")
    ax.set_title("Outlier Count by Feature (Tukey k=1.5)")

    for bar, (_, row) in zip(bars, outlier_df.iterrows()):
        ax.text(
            bar.get_width() + outlier_df["total_outliers"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{row['total_outliers']:,} ({row['outlier_pct']}%)",
            va="center", fontsize=9
        )

    plt.tight_layout()
    save_figure(fig, "phase2_outlier_summary")
    plt.show()

# Run
OUTLIER_COLS = [
    "fare_amount", "trip_distance", "tolls_amount",
    "tip_amount",  "extra",         "mta_tax",
    "improvement_surcharge", "passenger_count"
]

outlier_df = compute_iqr_outliers(df_raw, OUTLIER_COLS)

print("\n" + "=" * 70)
print("IQR Outlier Summary")
print("=" * 70)
print(outlier_df. to_string(index=False))

print("\n" + "=" * 70)
print("Extreme Fare Outlier Profiles (SQL)")
print("=" * 70)
extreme_fares = quick_sql("""
    SELECT
        CASE
            WHEN fare_amount > 100 THEN 'Very High (>$100)'
            WHEN fare_amount > 52 THEN 'High ($52-$100)'
            WHEN fare_amount < 0 THEN 'Negative'
            ELSE 'Normal'
        END AS fare_category,
        COUNT(*) AS trip_count,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        MODE(ratecodeid) AS most_common_ratecode
    FROM trips
    GROUP BY 1
    ORDER BY avg_fare DESC
""")
print(extreme_fares.to_string(index=False))

plot_outlier_summary(outlier_df)
plot_outlier_boxplots(df_raw, OUTLIER_COLS)


## Step 2.7: Outlier Profiling ✅

### IQR Outlier Summary (Tukey k=1.5)

| Column | Q1 | Q3 | IQR | Lower Fence | Upper Fence | Below | Above | Total | % |
|---|---|---|---|---|---|---|---|---|---|
| `trip_distance` | 1.00 | 3.30 | 2.30 | -2.45 | 6.75 | 0 | 112,031 | 112,031 | 11.20% |
| `passenger_count` | 1.00 | 2.00 | 1.00 | -0.50 | 3.50 | 0 | 101,929 | 101,929 | 10.19% |
| `fare_amount` | 6.50 | 14.50 | 8.00 | -5.50 | 26.50 | 117 | 96,370 | 96,487 | 9.65% |
| `tip_amount` | 0.00 | 2.26 | 2.26 | -3.39 | 5.65 | 1 | 59,799 | 59,800 | 5.98% |
| `tolls_amount` | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 6 | 57,451 | 57,457 | 5.75% |
| `mta_tax` | 0.50 | 0.50 | 0.00 | 0.50 | 0.50 | 5,987 | 35 | 6,022 | 0.60% |
| `extra` | 0.00 | 0.50 | 0.50 | -0.75 | 1.25 | 57 | 3,407 | 3,464 | 0.35% |
| `improvement_surcharge` | 0.30 | 0.30 | 0.00 | 0.30 | 0.30 | 921 | 1 | 922 | 0.09% |

### Extreme Fare Outlier Profiles

| Fare Category | Trips | Avg Distance | Avg Fare | Most Common Rate Code |
|---|---|---|---|---|
| Very High (>\\$100) | 777 | 22.60 miles | \\$151.10 | 5 (Negotiated) |
| High (\\$52–\\$100) | 7,123 | 17.02 miles | \\$66.07 | 1 (Standard) |
| Normal | 991,547 | 3.01 miles | \\$12.67 | 1 (Standard) |
| Negative | 553 | 0.29 miles | -\\$10.66 | 1 (Standard) |

---

### Plot Observations

**Image 1: Outlier Count by Feature (bar chart):**
- `trip_distance` leads with 112,031 outliers (11.20%), all above the upper fence of 6.75 miles. This is not noise but a real business
  pattern: airport runs, outer-borough trips, and negotiated fares systematically exceed the 6.75-mile fence. **These are legitimate   trips and should NOT be dropped**, the IQR fence is simply too narrow for a right-skewed distribution. log1p transformation in Phase 4 will handle the spread without removing valid data.
- `passenger_count` shows 101,929 outliers (10.19%), all above the upper fence of 3.5 passengers. Groups of 4, 5, and 6 are flagged
  purely because the IQR is only 1.0 (heavily solo-dominated). Again, these are legitimate trips, the IQR method is oversensitive here
  due to the near-constant Q1/Q3. Binning in Phase 4 resolves this.
- `fare_amount` has 96,487 outliers (9.65%), 96,370 above $26.50 and 117 below -$5.50. The upper outliers include valid high-value
  trips (airport flat rates, long-distance). The 117 below the lower fence are likely invalid negative fares already flagged in Phase 1.
- `tip_amount` and `tolls_amount` each have ~57–60K outliers (5–6%) driven by their zero-inflation, the IQR collapses to near-zero for
  `tolls_amount` (IQR=0.00), making every non-zero toll an "outlier". This is a statistical artefact of zero-inflated distributions, not true anomalies.
- `mta_tax`, `extra`, and `improvement_surcharge` each show fewer than 6,100 outliers (<0.6%), mostly the invalid negative values
  and rare non-standard charges already captured by Phase 1 contracts.

**Image 2: Box Plots (clipped p1–p99):**
- **`fare_amount`**: Compact IQR box (\\$6.50–\\$14.50) with a very long upper whisker extending to ~\\$27 and heavy outlier dots above, consistent with the right-skewed distribution seen in Step 2.1
- **`trip_distance`**: Similar pattern, tight box (1.0–3.3 miles) with a long upper whisker to ~7.5 miles and dense outlier cloud
  above, confirming the airport/long-trip tail
- **`tolls_amount`**: Extreme zero-inflation visible, the entire box collapses to zero (Q1=Q3=0), with the upper whisker and outlier
  cloud representing the minority of toll-paying trips. Every non-zero value is technically an "outlier" by IQR definition
- **`tip_amount`**: Tight box (0–\\$2.26) with a long whisker to ~\\$5.65 and a dense outlier cloud above, the zero spike from cash payments anchors Q1 at zero
- **`mta_tax`** and **`improvement_surcharge`**: Near-perfect flat lines, the entire value range is a single point (\\$0.50 and \\$0.30
  respectively), confirming near-zero variance and negligible predictive value
- **`extra`**: Small box (0–\\$0.50) with whisker to $1.00, the three valid surcharge values (\\$0, \\$0.50, \\$1.00) are visible as distinct clusters
- **`passenger_count`**: Compact box (1–2 passengers) with whisker to 3 and outlier dots at 4, 5, and 6, visually confirms that
  groups of 4+ are statistical outliers by IQR but are perfectly valid business trips

### Extreme Fare Analysis
- **Very High (>\\$100):** Only 777 trips (0.08\%) with avg distance of 22.60 miles, dominated by negotiated rate code 5. These are
  genuine long-distance or special-arrangement trips. Will be retained but monitored for model influence.
- **High (\\$52–\\$100):** 7,123 trips with avg distance 17.02 miles, predominantly standard rate code 1, likely outer-borough and
  airport-adjacent trips just above the JFK flat rate. Legitimate.
- **Negative fares:** 553 trips with avg distance of only 0.29 miles and avg fare of -$10.66, near-zero distance confirms these are
  meter errors or refund corrections, not real trips. All flagged for removal in Phase 3.

---

### Outlier Handling Strategy for Phase 3

| Column | Outlier Type | Volume | Action |
|---|---|---|---|
| `trip_distance` | Real long trips, IQR too narrow | 112,031 | **RETAIN**: log1p in Phase 4 |
| `passenger_count` | Groups of 4+, IQR oversensitive | 101,929 | **RETAIN**: bin in Phase 4 |
| `fare_amount` above fence | Valid high-value trips | 96,370 | **RETAIN**: log1p in Phase 4 |
| `fare_amount` below fence | Negative fares, invalid | 117 | **DROP**: Phase 1 contract |
| `tip_amount` | Zero-inflation artefact | 59,800 | **RETAIN**: log1p in Phase 4 |
| `tolls_amount` | Zero-inflation artefact | 57,457 | **RETAIN**: log1p in Phase 4 |
| `mta_tax` violations | Invalid values | 6,022 | **DROP**: Phase 1 contract |
| `extra` violations | Invalid surcharges | 3,464 | **DROP**: Phase 1 contract |
| Negative fares (extreme) | Meter errors/refunds | 553 | **DROP**: Phase 1 contract |

> **Key insight:** The IQR method significantly overstates the outlier
> problem for zero-inflated and right-skewed distributions. The true
> actionable outliers, negative fares, invalid surcharges, and meter
> errors, were already captured precisely by the value range contracts
> in Phase 1. Phase 3 will enforce those contracts rather than blindly
> applying IQR-based removal, which would discard 11% of legitimate
> long-distance trips.